# ABC Guide (PPS) for XMM-Newton -- EPIC Image Creation and Basic Filtering - Using PPS Files
<hr style="border: 2px solid #f5bf03" />

- **Description:** XMM-Newton ABC Guide, EPIC Image Creation and Basic Filtering, using PPS Files.
- **Level:** Beginner
- **Data:** XMM observation of the Lockman Hole (obsid=0123700101)
- **Requirements:** Must be run using pySAS version 2.3.0 or higher.
- **Credit:** Ryan Tanner (January 2026)
- **Support:** <a href="https://heasarc.gsfc.nasa.gov/docs/xmm/xmm_helpdesk.html">XMM Newton GOF Helpdesk</a>
- **Last verified to run:** 16 January 2026, for SAS v22.1 and pySAS v2.3.0

<hr style="border: 2px solid #f5bf03" />

## 1. Introduction
This tutorial is based on Chapter 7 from the The [The XMM-Newton ABC Guide](https://heasarc.gsfc.nasa.gov/docs/xmm/abc/ "ABC Guide") prepared by the NASA/GSFC XMM-Newton Guest Observer Facility. This notebook assumes you are at least minimally familiar with pySAS (see the [Long pySAS Introduction](./analysis-xmm-long-intro.ipynb "Long pySAS Intro")). 

<div class="alert alert-block alert-info">
    <b>Note:</b> This is a variation of the ABC Guide tutorial notebook on EPIC image creation and basic filtering. This tutorial uses the PPS files, while the other notebook uses the ODF files. See the notebook on PPS files for more information.
</div>

### A Brief Note About PPS File Names

PPS files have the following convention for their names:

```
POOOOOOOOOODDUEEETTTTTTSXXX.FFF

P          : All PPS Files start with a "P"
OOOOOOOOOO : Obs ID
DD         : Instrument/Product Identifier
U          : Exposure Flag
EEE        : Exposure Number
TTTTTT     : File/Product Type
S          : Data Subset Number/Character
XXX        : Source Number/Slew Step Number
FFF        : File Format
```

The Instrument/Product Identifier (DD) can take the values:

```
M1 : EPIC MOS 1 Camera
M2 : EPIC MOS 2 Camera
PN : EPIC PN Camera
EP : Combined EPIC Product
R1 : RGS 1 Camera
R2 : RGS 2 Camera
RG : Combined RGS Product
OM : Optical Monitor
CA : Catalogue Cross-Correlation File
OB : General Observation/Summary File
```

The Exposure Flag (U) can take the values:

```
S : Scheduled Exposure
U : Unscheduled Exposure
X : File Not an Exposure (General File)
```

The File or Product Type (TTTTTT) can have a number of values. We will not list them all here (there are > 50!), but we will note a few important ones.

```
MIEVLI : EPIC MOS Event List
PIEVLI : EPIC PN Event List
IMAGE_ : EPIC Image File
EVENLI : RGS Event List
SRSPEC : RGS Spectra File
SUMMAR : Observation Summary File
ATTTSR : Spacecraft Attitude File
CALIND : Calibration Index File
```

The files can be in the following formats:

```
FIT : FITS File
FTZ : Gzipped FITS File
HTM : HTML File
PNG : PNG File
PDF : PDF File
ASZ : Gzipped ASCII File
ASC : ASCII File
```

If the files are gzipped, you **do not** have to unzip them. SAS will handle that.

#### SAS Tasks to be Used

- `evselect`[(Documentation for evselect)](https://xmm-tools.cosmos.esa.int/external/sas/current/doc/evselect/index.html)
- `tabgtigen`[(Documentation for tabgtigen)](https://xmm-tools.cosmos.esa.int/external/sas/current/doc/tabgtigen/index.html)
- `gtibuild`[(Documentation for gtibuild)](https://xmm-tools.cosmos.esa.int/external/sas/current/doc/gtibuild/index.html)

#### Useful Links

- [`pysas` Documentation](https://xmm-tools.cosmos.esa.int/external/sas/current/doc/pysas/index.html "pysas Documentation")
- [`pysas` on GitHub](https://github.com/XMMGOF/pysas)
- [Common SAS Threads](https://www.cosmos.esa.int/web/xmm-newton/sas-threads/ "SAS Threads")
- [Users' Guide to the XMM-Newton Science Analysis System (SAS)](https://xmm-tools.cosmos.esa.int/external/xmm_user_support/documentation/sas_usg/USG/SASUSG.html "Users' Guide")
- [The XMM-Newton ABC Guide](https://heasarc.gsfc.nasa.gov/docs/xmm/abc/ "ABC Guide")
- [XMM Newton GOF Helpdesk](https://heasarc.gsfc.nasa.gov/docs/xmm/xmm_helpdesk.html "Helpdesk") - Link to form to contact the GOF Helpdesk.

<div class="alert alert-block alert-warning">
    <b>Warning:</b> By default this notebook will place observation data files in your default <tt>data_dir</tt> directory. Make sure pySAS has been configured properly.
</div>

## 2. Download PPS Files

In [ ]:
# pySAS imports
import pysas
from pysas import MyTask

# Useful imports
import os, glob, re
from IPython.display import HTML

# Imports for plotting
import matplotlib.pyplot as plt
from astropy.visualization import astropy_mpl_style
from astropy.io import fits
from astropy.wcs import WCS
from astropy.table import Table
plt.style.use(astropy_mpl_style)

# To handle certain warnings
import warnings
warnings.filterwarnings("ignore")

If you have already gone through the other tutorial that uses ODF files, what follows will be familiar, but slightly different. pySAS has an object class called `PPSFiles` that makes interacting with PPS files easier. For ODF files the equivalent object class is `ObsID`. They share almost all the same methods and information, but `PPSFiles` has methods specifically for PPS files.

If you work with the ODF files you first have to calibrate the data and then run the files through some basic processing using tasks such as `emproc` and `epproc` (or `emchain` and `epchain`). The event lists that come with the PPS files are already calibrated and have been passed through `emchain`, `epchain`, and `rgsproc`. So the first two steps of preparing the data for analysis are already done!

All we have to do is download the data. We do this by first creating a `PPSFiles` object using our desired Obs ID.

In [ ]:
obsid = '0123700101'
my_pps = pysas.PPSFiles(obsid)
my_pps.download_PPS_data(repo='heasarc',overwrite=False)

This will download all the PPS files associated with the Obs ID. Inside of the `my_pps` object there will be several variables that store the path and filename of key PPS files. The variable names are:

Strings containing the filename and path.
- `summary_file`
- `attitude_file`
- `calind_file`

Lists of strings containing the filenames and paths.
- `EPIC_event_lists`
- `EPIC_images`
- `RGS_event_lists`
- `RGS_spectra`

There are <tt>four</tt> EPIC event lists with this observation. There is one event list for the PN and MOS 2 cameras, but two event lists for the MOS 1 camera. One of those was "unscheduled" meaning during the collection of the data something inturrupted the data for the MOS 1 camera. To complete the data gathering, an additional, unscheduled, observation was made to make sure the MOS 1 had the same total time as the MOS 2 and pn cameras. This will not affect the analysis for this tutorial.

In [ ]:
print(f'Observation Summery File : {my_pps.summary_file}')
print(f'Attitude File            : {my_pps.attitude_file}')
print(f'Calibration Index File   : {my_pps.calind_file}')
print('\nEPIC Event Lists: my_pps.EPIC_event_lists')
for file in my_pps.EPIC_event_lists:
    print(f' > {file}')
print('\nEPIC FITS Image Files: my_pps.EPIC_images')
for file in my_pps.EPIC_images:
    print(f' > {file}')
print('\nRGS Event Lists: my_pps.RGS_event_lists')
for file in my_pps.RGS_event_lists:
    print(f' > {file}')
print('\nRGS FITS Spectra: my_pps.RGS_spectra')
for file in my_pps.RGS_spectra:
    print(f' > {file}')

## 3. Plot Images

From the list of image files we will select a few of them to display. We will use the regular expression `'.*(M1|M2|PN)S.*IMAGE_8.*.FTZ$'` to select the ones we want. This expression will select images from "scheduled" observations of the MOS 1, MOS 2, or pn. It will also only use the full band image. These images have already had some basic processing to remove the most common sources of noise.

In [ ]:
image_list = []

for file in my_pps.EPIC_images:
    if re.search('.*(M1|M2|PN)S.*IMAGE_8.*.FTZ$',file):
        image_list.append(file)

<div class="alert alert-block alert-info">
    <b>Note:</b> For this notebook we will be using a function named 'quick_implot' that is part of the 'PPSFiles' object for quick image plotting. It takes a FITS image file and plots it.
</div>

In [ ]:
for image in image_list:
    my_pps.quick_implot(image,vmin=0.01)

Let's compare these images to the raw (but calibrated) event lists. Event lists are **not** images, they are literally just a list of events registered by the detectors. To display an image of the events we need to convert them into an image. We do this using the SAS task `evselect`. Below we define a useful function to make image plotting easier. It uses `evselect` to create a FITS **image** file from a FITS event list file.

<div class="alert alert-block alert-info">
    <b>Note:</b> For this notebook we will be using a function named 'quick_eplot' that is part of the 'PPSFiles' object for quick image plotting. The equivelent code is shown below:
</div>

```python
def make_fits_image(event_list_file, image_file='image.fits'):
    
    inargs = {'table'        : event_list_file, 
              'withimageset' : 'yes',
              'imageset'     : image_file, 
              'xcolumn'      : 'X', 
              'ycolumn'      : 'Y', 
              'imagebinning' : 'imageSize', 
              'ximagesize'   : 600, 
              'yimagesize'   : 600}

    MyTask('evselect', inargs).run()

    hdu = fits.open(image_file)[0]
    wcs = WCS(hdu.header)

    ax = plt.subplot(projection=wcs)
    plt.imshow(hdu.data, origin='lower', norm='log', vmin=1.0, vmax=1e2)
    ax.set_facecolor("black")
    plt.grid(color='blue', ls='solid')
    plt.xlabel('RA')
    plt.ylabel('Dec')
    plt.colorbar()
    plt.show()
```

As a default `quick_eplot` creates a file named "image.fits" and this file will be overwritten each time the function is called. If you want your image file to have a unique name then use the function input `image_file`. For example:

```python
my_pps.quick_eplot('event_list_file.fits', image_file='my_special_image.fits')
```

---
The input arguments to `evselect` to create a FITS image file are:

    table - input event list file name
    withimageset - make an image
    imageset - name of output image file
    xcolumn - event column for X axis
    ycolumn - event column for Y axis
    imagebinning - form of binning, force entire image into a given size or bin by a specified number of pixels
    ximagesize - output image pixels in X
    yimagesize - output image pixels in Y

We also define a function to make plotting light curves simpler. As with the function `make_fits_image` it uses `evselect` to create the light curve and save it as a FITS file.

The input arguments to `evselect` to create a light curve file are:

    table - input event table
    withrateset - make a light curve
    rateset - name of output light curve file
    maketimecolumn - control to create a time column
    timecolumn - time column label
    timebinsize - time binning (seconds)
    makeratecolumn - control to create a count rate column, otherwise a count column will be created

<div class="alert alert-block alert-info">
    <b>Note:</b> For this notebook we will be using a function named 'quick_lcplot' that is part of the 'PPSFiles' object for quick light curve plotting. The equivelent code is shown below:
</div>

```python
def plot_light_curve(event_list_file, light_curve_file='ltcrv.fits'):
                     
    inargs = {'table'          : event_list_file, 
              'withrateset'    : 'yes', 
              'rateset'        : light_curve_file, 
              'maketimecolumn' : 'yes', 
              'timecolumn'     : 'TIME', 
              'timebinsize'    : '100', 
              'makeratecolumn' : 'yes'}

    MyTask('evselect', inargs).run()

    ts = Table.read(light_curve_file,hdu=1)
    plt.plot(ts['TIME'],ts['RATE'])
    plt.xlabel('Time (s)')
    plt.ylabel('Count Rate (ct/s)')
    plt.show()
```

We need to change into the work directory to run the next SAS tasks. We will also select the three scheduled exposures.

In [ ]:
event_lists = {}

for file in my_pps.EPIC_event_lists:
    if re.search('.*M1S.*.FTZ$',file):
        event_lists['EMOS1'] = file
    if re.search('.*M2S.*.FTZ$',file):
        event_lists['EMOS2'] = file
    if re.search('.*PNS.*.FTZ$',file):
        event_lists['EPN'] = file

event_lists

In [ ]:
os.chdir(my_pps.work_dir)

Here we plot an image of the raw data with no filters applied. The image should be very noisy.

In [ ]:
for key, evtli in event_lists.items():
    my_pps.quick_eplot(evtli,vmin=1.0,vmax=100.0)

## 4. Apply Standard Filter

To begin we apply a standard filter. The filtering expressions for the MOS and PN are, respectively:
```
(PATTERN $<=$ 12)&&(PI in [200:12000])&&#XMMEA_EM
```
and
```
(PATTERN $<=$ 4)&&(PI in [200:15000])&&#XMMEA_EP
```
The first two expressions will select good events with `PATTERN` in the 0 to 12 (or 0 to 4) range. The `PATTERN` value is similar the `GRADE` selection for ASCA data, and is related to the number and pattern of the CCD pixels triggered for a given event. The `PATTERN` assignments are: single pixel events: `PATTERN == 0`, double pixel events: `PATTERN in [1:4]`, triple and quadruple events: `PATTERN in [5:12]`.

The second keyword in the expressions, `PI`, selects the preferred pulse height of the event; for the MOS, this should be between 200 and 12000 eV. For the PN, this should be between 200 and 15000 eV. This should clean up the image significantly with most of the rest of the obvious contamination due to low pulse height events. Setting the lower `PI` channel limit somewhat higher (e.g., to 300 eV) will eliminate much of the rest.

Finally, the `#XMMEA_EM` (`#XMMEA_EP` for the PN) filter provides a canned screening set of `FLAG` values for the event. The `FLAG` value provides a bit encoding of various event conditions, e.g., near hot pixels or outside of the field of view. Setting `FLAG == 0` in the selection expression provides the most conservative screening criteria and should always be used when serious spectral analysis is to be done on the PN. It typically is not necessary for the MOS.

It is a good idea to keep the output filtered event files and use them in your analyses, as opposed to re-filtering the original file with every task. This will save much time and computer memory. As an example, the Lockman Hole data's original event file is 48.4 MB; the fully filtered list (that is, filtered spatially, temporally, and spectrally) is only 4.0MB!

The input arguments to `evselect` to apply the filter are:

    table - input event table
    filtertype - method of filtering
    expression - filtering expression
    withfilteredset - create a filtered set
    filteredset - output file name
    keepfilteroutput - save the filtered output
    updateexposure - update exposure information in event list and in spectrum files
    filterexposure - filter exposure extensions of event list with same time


In [ ]:
def filter_event_list(in_event_list,
                      filtered_event_list,
                      pi_min=500,
                      pi_max=10000):

    with fits.open(in_event_list) as hdu:
        instrument = hdu[0].header['INSTRUME']

    if instrument == 'EPN':
        filter = 'XMMEA_EP'
        pattern = 4
    elif 'EMOS' in instrument:
        filter = 'XMMEA_EM'
        pattern = 12

    # Filter expression
    expression = '(PATTERN in [0:{pattern}])&&(PI in [{pi_min}:{pi_max}])&&(FLAG == 0)&&#{filter}'.format(filter=filter,pattern=pattern,pi_min=pi_min,pi_max=pi_max)

    inargs = {'table'           : in_event_list, 
              'withfilteredset' : 'yes', 
              "expression"      : expression, 
              'filteredset'     : filtered_event_list, 
              'filtertype'      : 'expression', 
              'keepfilteroutput': 'yes', 
              'updateexposure'  : 'yes', 
              'filterexposure'  : 'yes'}
    
    MyTask('evselect', inargs).run()

In [ ]:
filtered_event_lists = {'EPN'   : 'EPN_filtered_events.fits',
                        'EMOS1' : 'EMOS1_filtered_events.fits',
                        'EMOS2' : 'EMOS2_filtered_events.fits'}

for key, evtli in event_lists.items():
    filter_event_list(evtli,filtered_event_lists[key],pi_min=500,pi_max=10000)

<div class="alert alert-block alert-info">
    <b>Note:</b> The expression for the input "<tt>expression</tt>" contains single quotes ('text'). The entire string needs to be surrounded by double quotes ("text") to preserve the single quotes inside the string. i.e. "This text has 'single quotes' inside of the double quotes."
</div>

Now we plot the filtered image. It should have less noise now.

In [ ]:
for key, evtli in filtered_event_lists.items():
    my_pps.quick_eplot(evtli,vmin=1.0,vmax=100.0)

## 5. Create Light Curve

<div class="alert alert-block alert-info">
    <b>Note:</b> For the following sections we will demonstrate filtering using ONLY the MOS 1 event list. The exact same methods can be done on the MOS 2 and pn event lists.
</div>

Sometimes, it is necessary to use filters on time in addition to those mentioned above. This is because of soft proton background flaring, which can have count rates of 100 counts/sec or higher across the entire bandpass. It should be noted that the amount of flaring that needs to be removed depends in part on the object observed; a faint, extended object will be more affected than a very bright X-ray source.

To see if background flaring should be removed we plot and examine the light curve.

In [ ]:
light_curve_file='EMOS1_ltcrv.fits'
my_pps.quick_lcplot(filtered_event_lists['EMOS1'],light_curve_file=light_curve_file)

Taking a look at the light curve, we can see that there is a very large flare toward the end of the observation and two much smaller ones in the middle of the exposure. Examining the light curve shows us that during non-flare times, the count rate is quite low, about 1.3 ct/s, with a small increase at 7.3223e7 seconds to about 6 ct/s. We can use that to further filter the data.

## 6. Applying Time or Rate Filters to the Data

There are many ways to filter the data. We will demonstrate four different methods. The first three methods will create a Good Time Interval (GTI) file which can then be used as an input to the command `evselect`. This will create a new, filtered, event list.

1. Create a secondary GTI file using the command `tabgtigen` and filter on `RATE`.
2. Create a secondary GTI file using the command `tabgtigen` and filter on `TIME`.
3. Create a *new* GTI file using the command `gtibuild` and filter on `TIME`.
4. Filter on `TIME` using an explicit reference in the inputs to the command `evselect`.

For the last method the user explicitly inputs the time intervals to be used as an expression for the command `evselect` rather than using a separate GTI file. All of these will get the job done, so which to use is a matter of the user's preference.

### 6.1 Using `tabgtigen` to filter on `RATE`

The inputs for `tabgtigen` are:

    table - input file name with count rate table
    gtiset - output file name for selected GTI intervals
    timecolumn - time column
    expression - filtering expression
    
We choose a rate $<= 6$ counts/s and filter based on that. As the input we use the lightcurve file created in Section 5.

In [ ]:
gti_rate_file = 'gti_rate.fits'
mos1_filt_rate = 'EMOS1_filt_rate.fits'

inargs = {'table'      : light_curve_file, 
          'gtiset'     : gti_rate_file,
          'timecolumn' : 'TIME', 
          "expression" : "'(RATE <= 6)'"}

MyTask('tabgtigen', inargs).run()

inargs = {'table'           : filtered_event_lists['EMOS1'],
          'withfilteredset' : 'yes', 
          "expression"      : "'GTI({0},TIME)'".format(gti_rate_file), 
          'filteredset'     : mos1_filt_rate,
          'filtertype'      : 'expression', 
          'keepfilteroutput': 'yes',
          'updateexposure'  : 'yes', 
          'filterexposure'  : 'yes'}

MyTask('evselect', inargs).run()

Now we create an image from the new event list that has been filtered based on `RATE`. There should be significantly less noise and only point sources should remain. Compare this final image to the first raw, unfilted image.

In [ ]:
my_pps.quick_eplot(mos1_filt_rate, image_file='final_image1.fits')

We can also create a new light curve from the filtered event list and compare it to the light curve from Section 5 to see what we have done.

In [ ]:
my_pps.quick_lcplot(mos1_filt_rate)

### 6.2 Using `tabgtigen` to filter on `TIME`

Alternatively, we could have chosen to make a new GTI file by noting the times of the flaring in the light curve and using that as a filtering parameter. The big flare starts around 7.32276e7 s, and the smaller ones are at 7.32119e7 s and 7.32205e7 s. The expression to remove these would be `(TIME <= 73227600)&&!(TIME IN [7.32118e7:7.3212e7])&&!(TIME IN [7.32204e7:7.32206e7])`. The syntax `(TIME <= 73227600)` includes only events with times less than or equal to `73227600`, and the "!" symbol stands for the logical "not", so use `&&!(TIME in [7.32118e7:7.3212e7])` to exclude events in that time interval. Once the new GTI file is made, we apply it with `evselect`. Everything else remains the same as in Section 6.1.

In [ ]:
gti_time_file = 'gti_rate.fits'
mos1_filt_time = 'EMOS1_filt_time.fits'

inargs = {'table'      : light_curve_file, 
          'gtiset'     : gti_time_file,
          'timecolumn' : 'TIME', 
          "expression" : "'(TIME <= 73227600)&&!(TIME IN [7.32118e7:7.3212e7])&&!(TIME IN [7.32204e7:7.32206e7])'"}

MyTask('tabgtigen', inargs).run()

inargs = {'table'           : filtered_event_lists['EMOS1'],
          'withfilteredset' : 'yes', 
          "expression"      : "'GTI({0},TIME)'".format(gti_rate_file), 
          'filteredset'     : mos1_filt_time,
          'filtertype'      : 'expression', 
          'keepfilteroutput': 'yes',
          'updateexposure'  : 'yes', 
          'filterexposure'  : 'yes'}

MyTask('evselect', inargs).run()

We can now plot the image that has been filtered on `TIME` and compare it to the image that was been filtered on `RATE` from above.

In [ ]:
my_pps.quick_eplot(mos1_filt_time, image_file='final_image2.fits')
my_pps.quick_lcplot(mos1_filt_time)

### 6.3 Using `gtibuild` to make a new GTI file and filter on `TIME`

This method requires a text file as input. The file should be in ASCII format with eash row on a new line and values for each column separated by spaces. In the first two columns, enter the start and end times (in seconds) that you are interested in, and in the third column, indicate with either a + or - sign whether that region should be kept or removed. Each good (or bad) time interval should get its own line, with any optional comments preceeded by a "#". In the example case, we would write in our ASCII file (named gti.txt):

In [ ]:
gti_txt_file = 'gti.txt'

gti_lines = ['0        73227600 + # Good time from the start of the observation',
             '73211800 73212000 - # But without a small flare here.',
             '73220400 73220600 - # And here.']

with open(gti_txt_file, 'w') as f:
    f.writelines(gti_lines)

We can now run `gtibuild` to create a new GTI file.

---
The inputs for `gtibuild` are:

    file - input text file name
    table - output GTI file name


In [ ]:
new_gti_file = 'new_gti.fits'

inargs = {'file'  : gti_txt_file,
          'table' : new_gti_file}

MyTask('gtibuild', inargs).run()

We can now run `evselect` as before with the new GTI file.

In [ ]:
mos1_new_gti = 'EMOS1_new_gti.fits'

inargs = {'table'           : filtered_event_lists['EMOS1'],
          'withfilteredset' : 'yes', 
          "expression"      : "'GTI({0},TIME)'".format(new_gti_file), 
          'filteredset'     : mos1_new_gti,
          'filtertype'      : 'expression', 
          'keepfilteroutput': 'yes',
          'updateexposure'  : 'yes', 
          'filterexposure'  : 'yes'}

MyTask('evselect', inargs).run()

If you want, you can compare the new image and light curve to what was made previously.

In [ ]:
my_pps.quick_eplot(mos1_new_gti, image_file='final_image3.fits')
my_pps.quick_lcplot(mos1_new_gti)

### 6.4 Filter on `TIME` by Explicit Reference

Finally, we could have chosen to forgo making a secondary GTI file altogether, and simply filtered on `TIME` with the standard filtering expression (see Section 4). The filtering expression from Section 4 can be combined with the filtering expression from Section 6.2 and filter the raw data all in one step. In this case, the full filtering expression would be:

In [ ]:
expression = "'(PATTERN <= 12)&&(PI in [200:12000])&&#XMMEA_EM&&(TIME <= 73227600) &&!(TIME IN [7.32118e7:7.3212e7])&&!(TIME IN [7.32204e7:7.32206e7])'"

and we would run `evselect` as the same way we did in Section 4.

In [ ]:
full_filt_event_list = 'EMOS1_filt.fits'

inargs = {'table'           : event_lists['EMOS1'],
          'withfilteredset' : 'yes', 
          "expression"      : expression, 
          'filteredset'     : full_filt_event_list,
          'filtertype'      : 'expression', 
          'keepfilteroutput': 'yes',
          'updateexposure'  : 'yes', 
          'filterexposure'  : 'yes'}

MyTask('evselect', inargs).run()

Finally we can compare the result with what we made before.

In [ ]:
my_pps.quick_eplot(full_filt_event_list, image_file='final_image4.fits')
my_pps.quick_lcplot(full_filt_event_list)

## 7. Conclusion

We have demonstrated various filtering techniques to remove noise from the raw observation data. Note: How you filter on `RATE` or `TIME` will depend on the light curve of each individual observation. For exceptionally bright sources you may only have to apply the standard filter.

To continue on from here, the next Jupyter Notebook in the series covers [source detection, spectra extraction, pile up, and preparing the spectra for analysis](./analysis-xmm-ABC-PPS-guide-EPIC-source-spectrum.ipynb) by creating a redistribution matrix file (RMF) and an ancillary response file (ARF).

Below we have included a short script that incorporates all of the filtering steps for a single observation for MOS1, but without making any plots or image files. 

```python
obsid = '0123700101'
my_pps = pysas.PPSFiles(obsid)
my_pps.download_PPS_data(repo='heasarc',overwrite=False)

os.chdir(my_pps.work_dir)
for file in my_pps.EPIC_event_lists:
    if re.search('.*M1S.*.FTZ$',file):
        unfiltered_event_list = file

# The User can change these file names
temporary_event_list = 'temporary_event_list.fits' # Created by the "standard" filter
light_curve_file = 'mos1_ltcrv.fits'               # Light curve file name
gti_rate_file = 'gti_rate.fits'                    # GTI file name
filtered_event_list = 'filtered_event_list.fits'   # Final filtered 

# "Standard" Filter
inargs = {'table'           : unfiltered_event_list, 
          'withfilteredset' : 'yes', 
          "expression"      : "'(PATTERN <= 12)&&(PI in [200:4000])&&#XMMEA_EM'", 
          'filteredset'     : temporary_event_list, 
          'filtertype'      : 'expression', 
          'keepfilteroutput': 'yes', 
          'updateexposure'  : 'yes', 
          'filterexposure'  : 'yes'}

MyTask('evselect', inargs).run()

# Make Light Curve File
inargs = {'table'          : temporary_event_list, 
          'withrateset'    : 'yes', 
          'rateset'        : light_curve_file, 
          'maketimecolumn' : 'yes', 
          'timecolumn'     : 'TIME', 
          'timebinsize'    : '100', 
          'makeratecolumn' : 'yes'}

MyTask('evselect', inargs).run()

# Make Secondary GTI File
# Chose the rate based on the plot from the light curve file
filter_rate = 6
inargs = {'table'      : light_curve_file, 
          'gtiset'     : gti_rate_file,
          'timecolumn' : 'TIME', 
          "expression" : "'(RATE <= {0})'".format(filter_rate)}

MyTask('tabgtigen', inargs).run()

# Filter Using Secondary GTI File
inargs = {'table'           : temporary_event_list,
          'withfilteredset' : 'yes', 
          "expression"      : "'GTI({0},TIME)'".format(gti_rate_file), 
          'filteredset'     : filtered_event_list,
          'filtertype'      : 'expression', 
          'keepfilteroutput': 'yes',
          'updateexposure'  : 'yes', 
          'filterexposure'  : 'yes'}

MyTask('evselect', inargs).run()
```